In [0]:
import dlt
from pyspark.sql.functions import current_timestamp
 
@dlt.table(
    name="spl_bronze",
    comment="Raw Seattle Pet License data with metadata from cloud_files.",
    table_properties={
        "checkpointLocation": "/Volumes/workspace/damg7370/datastore/SPL/checkpoint/spl_bronze",
        "delta.columnMapping.mode": "name"
    }
)
def spl_bronze():
    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .load("/Volumes/workspace/damg7370/datastore/SPL/")
    )
 
    # Add load time and file metadata
    df = (
        df.withColumn("load_dt", current_timestamp())
          .withColumn("_source_file_path", df["_metadata.file_path"])
          .withColumn("_source_file_name", df["_metadata.file_name"])
    )
    return df

In [0]:
from pyspark.sql.functions import col, substring, to_date, concat, lit, last_day, date_format
 
@dlt.table(
    name="spl_cleansed_silver",
    comment="Cleansed Seattle Pet License data with derived date attributes and null checks."
)
def spl_cleansed_silver():
    df = dlt.read_stream("spl_bronze")
 
    # Rename and select relevant columns
    df = df.select(
        col("License Number").alias("license_number"),
        col("Animal's Name").alias("animal_name"),
        col("Species").alias("species"),
        col("Primary Breed").alias("primary_breed"),
        col("Secondary Breed").alias("secondary_breed"),
        col("ZIP Code").alias("zip_code"),
        col("load_dt"),
        col("_source_file_name")
    )
 
    # Extract year/month from file name (e.g., Seattle_Pet_Licenses_202107.csv)
    df = df.withColumn("file_year", substring(col("_source_file_name"), 24, 4).cast("int"))
    df = df.withColumn("file_month", substring(col("_source_file_name"), 28, 2).cast("int"))
 
    # Compute end of month and source_file_date (yyyyMMdd)
    df = df.withColumn(
        "end_date",
        last_day(to_date(concat(substring(col("_source_file_name"), 24, 6), lit("01")), "yyyyMMdd"))
    )
    df = df.withColumn("source_file_date", date_format(col("end_date"), "yyyyMMdd"))
 
    # Drop records with null license_number or zip_code
    df = df.filter(col("license_number").isNotNull() & col("zip_code").isNotNull())
 
    return df

In [0]:
@dlt.table(
    name="spl_species_breed_type_silver",
    comment="Distinct combination of species and breed types for reference lookups."
)
def spl_species_breed_type_silver():
    df = dlt.read_stream("spl_cleansed_silver")
    df = df.select("species", "primary_breed", "secondary_breed", "load_dt").distinct()
    return df

In [0]:
import dlt
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window
 
@dlt.table(
    name="spl_species_dim",
    comment="Species dimension table with surrogate key for species-breed combinations."
)
def spl_species_dim():
    df = dlt.read("spl_species_breed_type_silver")
    df = df.withColumn("species_key", row_number().over(Window.orderBy("species")))
    return df.select("species_key", "species", "primary_breed", "secondary_breed")

In [0]:
import dlt

from pyspark.sql import functions as F

from datetime import date
 
@dlt.table(

    name="date_dim",

    comment="Date dimension table generated safely from Silver data or defaults to a fixed range."

)

def date_dim():

    silver_df = dlt.read("spl_cleansed_silver")
 
    # Get min and max dates from Silver

    date_range = silver_df.select(

        F.min(F.to_date("source_file_date", "yyyyMMdd")).alias("min_dt"),

        F.max(F.to_date("source_file_date", "yyyyMMdd")).alias("max_dt")

    ).collect()[0]
 
    start_date = date_range["min_dt"]

    end_date = date_range["max_dt"]
 
    # Fallback if Silver not loaded yet

    if start_date is None or end_date is None:

        start_date = date(2021, 1, 1)

        end_date = date(2021, 12, 31)
 
    # Calculate total number of days between start and end dates

    num_days = (end_date - start_date).days + 1
 
    #Convert BIGINT id → INT for date_add() to avoid type mismatch

    df = (

        spark.range(0, num_days)

        .withColumn("id_int", F.col("id").cast("int"))

        .withColumn("date", F.expr(f"date_add('{start_date}', id_int)"))

        .withColumn("date_key", F.date_format("date", "yyyyMMdd"))

        .withColumn("year", F.year("date"))

        .withColumn("month", F.month("date"))

        .withColumn("day", F.dayofmonth("date"))

        .withColumn("month_name", F.date_format("date", "MMMM"))

        .withColumn("quarter", F.quarter("date"))

    )
 
    return df.select("date_key", "date", "year", "month", "month_name", "quarter", "day")

 

In [0]:
@dlt.table(
    name="geo_dim",
    comment="Geo dimension table joined to Silver layer for DLT lineage tracking."
)
def geo_dim():
    silver_df = dlt.read("spl_cleansed_silver")  # adds dependency
    csv_path = "/Volumes/workspace/damg7370/datastore/geo-data.csv"

    geo_df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(csv_path)
        .withColumnRenamed("zipcode", "zip_code")
        .withColumn("geo_key", F.monotonically_increasing_id())
        .select("geo_key", "zip_code", "city", "state", "county", "state_fips")
    )

    #Join not required — this just registers dependency
    _ = silver_df.limit(1)

    return geo_df


In [0]:
import dlt

from pyspark.sql import functions as F
 
@dlt.table(

    name="spl_fact_pet_license",

    comment="Fact table joining Silver data with Date, Geo, and Species dimensions at license-level granularity."

)

def spl_fact_pet_license():

    # Read all relevant tables

    s = dlt.read("spl_cleansed_silver")

    d = dlt.read("date_dim")

    g = dlt.read("geo_dim")

    sp = dlt.read("spl_species_dim")
 
    # Join with dimensions

    fact_df = (

        s.join(d, s["source_file_date"] == d["date_key"], "left")

         .join(

             sp,

             (s["species"] == sp["species"]) &

             (s["primary_breed"] == sp["primary_breed"]) &

             (s["secondary_breed"] == sp["secondary_breed"]),

             "left"

         )

         .join(g, s["zip_code"] == g["zip_code"], "left")

    )
 
    # Select fact-level attributes and dimension keys

    fact_df = fact_df.select(

        d["date_key"],

        sp["species_key"],

        g["geo_key"],

        s["license_number"],

        s["animal_name"],

        s["zip_code"],

        s["species"],

        s["primary_breed"],

        s["secondary_breed"],

        s["file_year"],

        s["file_month"],

        s["load_dt"]

    )
 
    # Add derived columns if needed (example: year_month concatenation)

    fact_df = fact_df.withColumn(

        "year_month",

        F.concat_ws("-", F.col("file_year").cast("string"), F.lpad(F.col("file_month"), 2, "0"))

    )
 
    return fact_df

 